# Part 1. Research Area Map

This notebook draws the South Island overview and Fox Glacier inset, then saves `south_island_dem.png` in the current folder.

Keep this notebook, the other lab files, and `fox_glacier_from_osm.geojson` together in the same folder. Intermediate CPT/NC files are written to a temporary directory, not to the lab folder.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np

# Help PyGMT find GMT and avoid PROJ database conflicts in notebook kernels.
for _gmt_lib in (
    "/usr/lib/x86_64-linux-gnu/libgmt.so",
    "/usr/lib/x86_64-linux-gnu/libgmt.so.6",
):
    if os.path.exists(_gmt_lib):
        os.environ.setdefault("GMT_LIBRARY_PATH", _gmt_lib)
        break
try:
    import pyproj

    _proj_data = pyproj.datadir.get_data_dir()
    os.environ["PROJ_DATA"] = _proj_data
    os.environ["PROJ_LIB"] = _proj_data
except Exception:
    pass

import pygmt


def ensure_conda_env_runtime() -> None:
    """Align Jupyter’s environment with what ``conda activate`` would provide.

    Many notebook kernels start Python without modifying ``PATH``, so bare subprocess
    calls to ``gmt`` / ``gdalwarp`` fail even though those binaries exist under the same
    conda env as ``sys.executable``. Similarly, ``gdalwarp`` needs PROJ/GDAL resource
    directories; without ``PROJ_DATA`` / ``GDAL_DATA``, reprojection can error with
    invalid CRS even when EPSG:4326 is requested.

    We prepend ``.../envs/<name>/bin`` to ``PATH`` and set data dirs under
    ``.../envs/<name>/share`` only if unset, so user overrides still win.
    """
    bindir = Path(sys.executable).resolve().parent  # e.g. .../RSA/bin
    prefix = bindir.parent  # env root
    path = os.environ.get("PATH", "")
    bd = str(bindir)
    if bd not in path.split(os.pathsep):
        os.environ["PATH"] = bd + os.pathsep + path
    proj = prefix / "share" / "proj"
    if proj.is_dir():
        os.environ.setdefault("PROJ_DATA", str(proj))
        os.environ.setdefault("PROJ_LIB", str(proj))
    gdal_data = prefix / "share" / "gdal"
    if gdal_data.is_dir():
        os.environ.setdefault("GDAL_DATA", str(gdal_data))


ensure_conda_env_runtime()

# Enable GMT remote datasets. Codespaces starts from a clean GMT cache, so the
# Earth relief grid must be downloaded the first time this notebook runs.
os.environ["GMT_DATA_UPDATE_INTERVAL"] = "1d"
subprocess.run(["gmt", "set", "GMT_DATA_UPDATE_INTERVAL", "1d"], check=True)

# All inputs and outputs stay in the current lab folder.
ROOT = Path.cwd().resolve()
_TEMP_DIR = tempfile.TemporaryDirectory(prefix="fox_map_")
TMP_ROOT = Path(_TEMP_DIR.name)

# ---------------------------------------------------------------------------
# Regions & projections (single source of truth for boxes and tiles)
# ---------------------------------------------------------------------------
# Left overview: full South Island context in WGS84 [west, east, south, north].
R_LEFT = [166.0, 174.5, -47.5, -40.3]
# Fixed Mercator width ``M15c`` → GMT chooses height from latitude span.
J_LEFT = "M15c"

# Inset bounds for panel B AND for tile download/warp. Must equal the dashed polygon
# drawn on panel A (see fx/fy in drawing cell).
FOX_W, FOX_E, FOX_S, FOX_N = 170.0, 170.2, -43.62, -43.42
FOX_REGION = [FOX_W, FOX_E, FOX_S, FOX_N]

# Raw mosaic from contextily remains in Web Mercator (EPSG:3857); warped copy is WGS84.
FOX_RAW_TIF = ROOT / "fox_basemap_raw.tif"
FOX_PLOT_TIF = ROOT / "fox_basemap_plot.tif"
TILE_ZOOM = 14  # Higher zoom → sharper but larger download and slower warp.

# Horizontal gap (cm) between panels on the composite canvas.
GAP_CM = 0.45
# Eastern edge reference points of FOX_REGION (used with mapproject for connectors).
BOX_TR = (FOX_E, FOX_N)
BOX_BR = (FOX_E, FOX_S)

# Fractional inset from SW corner for panel letters “A” / “B” so labels sit inside frame.
LABEL_PAD_X_FRAC = 0.015
LABEL_PAD_Y_FRAC = 0.017

# Smaller polygon inside panel B highlighting glacier extent (outline only).
FOX_GL_RNG_W, FOX_GL_RNG_E = 170.06, 170.167
FOX_GL_RNG_S, FOX_GL_RNG_N = -43.56, -43.48

# GMT basemap embellishments (see GMT docs for ``map_scale`` / ``rose`` syntax).
LEFT_FONT_ANNOT = "10p"
LEFT_MAP_SCALE = "jBR+w200k+o0.65c/0.75c"
LEFT_ROSE = "jTL+w1.55c+l"
RIGHT_MAP_SCALE = "jBR+w2k+o0.65c/0.75c"
# Second basemap pass uses white pens/fonts so the rose reads on bright imagery.
RIGHT_ROSE_WHITE = "jTL+w1.55c+l+p1p,white"

# Esri attribution box behind text (GMT ``text`` fill + clearance + optional outline pen).
BASEMAP_CREDIT_FILL = "gray@25"
BASEMAP_CREDIT_CLEARANCE = "0.14c"
BASEMAP_CREDIT_BOX_PEN = "0.2p,#d8d8d0"

OUT_PNG = ROOT / "south_island_dem.png"
CPT_L = TMP_ROOT / "south_island_relief.cpt"
LABEL_PANEL = "18p,Helvetica-Bold,black"



In [ ]:
# =============================================================================
# Helpers: GMT subprocess wrappers & imagery I/O
# =============================================================================

def panel_label_corner_lon_lat(region: list[float]) -> tuple[float, float]:
    """Return lon/lat for panel labels A/B near the lower-left of ``region``.

    GMT ``text`` uses map coordinates; placing labels at a fixed fraction of the span
    from the southwestern corner keeps typography consistent when regions change.
    """
    w, e, s, n = region[0], region[1], region[2], region[3]
    lon = w + LABEL_PAD_X_FRAC * (e - w)
    lat = s + LABEL_PAD_Y_FRAC * (n - s)
    return lon, lat


def _rstr(r: list[float]) -> str:
    """Format ``region`` as GMT ``-R`` string ``west/east/south/north``."""
    return "/".join(map(str, r))


def map_dims_cm(region: list[float], projection: str) -> tuple[float, float]:
    """Return map width and height in centimeters for ``-Rregion`` and ``-Jprojection``.

    Uses ``gmt mapproject -W``, which reports the bounding box size of the projected
    map—needed to reserve matching vertical space for panel B and for absolute ``X`` plots.
    """
    out = subprocess.run(
        ["gmt", "mapproject", "-R" + _rstr(region), "-J" + projection, "-W"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()
    return float(out[0]), float(out[1])


def mapproject_cm(lon: float, lat: float, region: list[float], projection: str) -> tuple[float, float]:
    """Project one geographic point into map coordinates (cm) for ``region`` + ``projection``.

    Feed corners here to obtain positions inside each panel’s local plot space; later we
    offset panel B using ``shift_origin`` so connector endpoints align on the composite.
    """
    out = subprocess.run(
        ["gmt", "mapproject", "-R" + _rstr(region), "-J" + projection],
        input=f"{lon} {lat}\n",
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()
    return float(out[0]), float(out[1])


def mercator_J_matching_height(region: list[float], target_h_cm: float, tol: float = 0.02) -> tuple[str, float, float]:
    """Pick Mercator ``-JMwidth`` so panel B height matches ``target_h_cm``.

    PyGMT stacks panels side-by-side; matching heights avoids awkward whitespace. Because
    Mercator height grows nonlinearly with ``width``, we binary-search the ``M{width}c``
    string until ``map_dims_cm`` reports the desired height within ``tol`` cm.
    """
    lo, hi = 1.0, 120.0
    mid = 15.0
    for _ in range(56):
        mid = (lo + hi) / 2
        w_cm, h_cm = map_dims_cm(region, f"M{mid}c")
        if abs(h_cm - target_h_cm) < tol:
            return f"M{mid}c", w_cm, h_cm
        if h_cm < target_h_cm:
            lo = mid
        else:
            hi = mid
    w_cm, h_cm = map_dims_cm(region, f"M{mid}c")
    return f"M{mid}c", w_cm, h_cm


def download_tile_basemap(region: list[float], path: Path, zoom: int = TILE_ZOOM) -> None:
    """Download Esri World Imagery XYZ tiles covering ``region`` into ``path``.

    ``bounds2raster`` writes a GeoTIFF georeferenced in Web Mercator (EPSG:3857). That
    CRS differs slightly from GMT’s ellipsoidal Mercator used by ``-JM``, so we still
    warp to geographic lon/lat before ``grdimage``.
    """
    import contextily as cx

    w, e, s, n = region[0], region[1], region[2], region[3]
    cx.bounds2raster(w, s, e, n, str(path), ll=True, zoom=zoom, source=cx.providers.Esri.WorldImagery)


def warp_tiles_to_wgs84_plot_grid(src: Path, dst: Path, region: list[float]) -> None:
    """Warp ``src`` (3857 mosaic) to WGS84 grid exactly filling ``region``.

    ``-te west south east north`` pins output bounds to ``FOX_REGION``. Cubic resampling
    keeps imagery smooth; LZW shrinks file size. This step exists because PyGMT should use
    ``grdimage`` on a geographic grid + ``-JM``—GMT ``image`` treats pixels linearly in
    lon/lat and will not align with the Mercator frame used by ``basemap``.
    """
    w, e, s, n = region[0], region[1], region[2], region[3]
    subprocess.run(
        ["gdalwarp", "-overwrite", "-t_srs", "EPSG:4326", "-te", str(w), str(s), str(e), str(n),
         "-r", "cubic", "-co", "COMPRESS=LZW", str(src), str(dst)],
        check=True,
    )



In [ ]:
try:
    with pygmt.config(GMT_DATA_UPDATE_INTERVAL="1d"):
        grid_l = pygmt.datasets.load_earth_relief(resolution="30s", region=R_LEFT)
except Exception:
    # If an existing GMT session still has remote downloads disabled, prefetch the
    # relief grid through the GMT CLI after resetting the same default, then reload.
    subprocess.run(["gmt", "set", "GMT_DATA_UPDATE_INTERVAL", "1d"], check=True)
    subprocess.run(["gmt", "which", "-Gc", "@earth_relief_30s_g"], check=True)
    with pygmt.config(GMT_DATA_UPDATE_INTERVAL="1d"):
        grid_l = pygmt.datasets.load_earth_relief(resolution="30s", region=R_LEFT)
land_l = grid_l.where(grid_l >= 0)
zv = land_l.values[np.isfinite(land_l.values)]
if zv.size == 0:
    raise RuntimeError("No land pixels (overview)")
zmin, zmax = float(np.nanmin(zv)), float(np.nanmax(zv))
pygmt.makecpt(
    cmap="#ffffff,#fafafa,#f5f5f5,#ebebeb,#e2e2e2,#d8d8d8",
    series=[zmin, zmax, max((zmax - zmin) / 256, 1e-6)],
    continuous=True,
    output=str(CPT_L),
)
shade_l = pygmt.grdgradient(grid=grid_l, azimuth=315, normalize="t")

if not FOX_RAW_TIF.is_file():
    download_tile_basemap(FOX_REGION, FOX_RAW_TIF)
if not FOX_PLOT_TIF.is_file():
    warp_tiles_to_wgs84_plot_grid(FOX_RAW_TIF, FOX_PLOT_TIF, FOX_REGION)

w1, h1 = map_dims_cm(R_LEFT, J_LEFT)
J_RIGHT, w2, h2 = mercator_J_matching_height(FOX_REGION, h1)

W = w1 + GAP_CM + w2
H = h1
y_shift_cm = 0.0

sx1, sy1 = mapproject_cm(*BOX_TR, R_LEFT, J_LEFT)
sx2, sy2 = mapproject_cm(*BOX_BR, R_LEFT, J_LEFT)
lon_w, lat_s, lat_n = FOX_REGION[0], FOX_REGION[2], FOX_REGION[3]
ex1, ey1 = mapproject_cm(lon_w, lat_n, FOX_REGION, J_RIGHT)
ex2, ey2 = mapproject_cm(lon_w, lat_s, FOX_REGION, J_RIGHT)
xg1 = w1 + GAP_CM + ex1
xg2 = w1 + GAP_CM + ex2
yg1 = y_shift_cm + ey1
yg2 = y_shift_cm + ey2

fig = pygmt.Figure()

fig.grdimage(grid=land_l, cmap=str(CPT_L), shading=shade_l, region=R_LEFT, projection=J_LEFT)
fig.coast(region=R_LEFT, projection=J_LEFT, water="#87CEEB", shorelines="0.1p,#888888")

fx = [FOX_W, FOX_E, FOX_E, FOX_W, FOX_W]
fy = [FOX_S, FOX_S, FOX_N, FOX_N, FOX_S]
fig.plot(x=fx, y=fy, pen="1.5p,black")
fig.text(x=170.10, y=-43.3, text="Fox Glacier", font="12p,Helvetica-Bold,black", justify="CB")

fig.plot(x=[172.6307], y=[-43.5321], style="c0.22c", pen="0.35p,black", fill="white")
fig.text(x=172.6307, y=-43.70, text="Christchurch", font="12p,Helvetica-Bold,black", justify="CT")

fig.text(x=167.75, y=-45.0, text="Southern Alps", font="17p,Helvetica-Bold,black", angle=52, justify="CM")
fig.text(x=168.85, y=-41.85, text="New Zealand", font="18p,Helvetica-Bold,black", justify="CM")
fig.text(x=168.85, y=-42.38, text="South Island", font="16p,Helvetica-Bold,black", justify="CM")
fig.text(x=172.6, y=-46.0, text="Pacific Ocean", font="15p,Helvetica-Bold,#003366", justify="CM")

with pygmt.config(MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY=LEFT_FONT_ANNOT):
    fig.basemap(
        region=R_LEFT,
        projection=J_LEFT,
        frame=["af"],
        map_scale=LEFT_MAP_SCALE,
        rose=LEFT_ROSE,
    )

lon_a, lat_a = panel_label_corner_lon_lat(R_LEFT)
fig.text(x=lon_a, y=lat_a, text="A", font=LABEL_PANEL, justify="LB")

with fig.shift_origin(xshift=f"{w1 + GAP_CM}c", yshift=f"{y_shift_cm}c"):
    fig.grdimage(grid=str(FOX_PLOT_TIF), region=FOX_REGION, projection=J_RIGHT)

    gx = [FOX_GL_RNG_W, FOX_GL_RNG_E, FOX_GL_RNG_E, FOX_GL_RNG_W, FOX_GL_RNG_W]
    gy = [FOX_GL_RNG_S, FOX_GL_RNG_S, FOX_GL_RNG_N, FOX_GL_RNG_N, FOX_GL_RNG_S]
    fig.plot(x=gx, y=gy, pen="1.5p,yellow", region=FOX_REGION, projection=J_RIGHT)
    fig.text(
        x=0.5 * (FOX_GL_RNG_W + FOX_GL_RNG_E) - 0.02,
        y=0.5 * (FOX_GL_RNG_S + FOX_GL_RNG_N) - 0.03,
        text="Fox Glacier",
        font="14p,Helvetica-Bold,yellow",
        justify="CM",
        region=FOX_REGION,
        projection=J_RIGHT,
    )

    with pygmt.config(
        MAP_FRAME_TYPE="plain",
        FONT_ANNOT_PRIMARY=LEFT_FONT_ANNOT,
        MAP_TICK_LENGTH_PRIMARY="0p",
        MAP_TICK_LENGTH_SECONDARY="0p",
    ):
        fig.basemap(
            region=FOX_REGION,
            projection=J_RIGHT,
            frame=["wsne"],
            map_scale=RIGHT_MAP_SCALE,
        )
    with pygmt.config(
        FONT_ANNOT_PRIMARY="10p,Helvetica,white",
        FONT_LABEL="10p,Helvetica-Bold,white",
        FONT_TITLE="10p,Helvetica,white",
        MAP_DEFAULT_PEN="1p,white",
        MAP_TICK_PEN_PRIMARY="0.5p,white",
        MAP_TICK_PEN_SECONDARY="0.5p,white",
        MAP_TITLE_OFFSET="0p",
    ):
        fig.basemap(region=FOX_REGION, projection=J_RIGHT, frame=["n"], rose=RIGHT_ROSE_WHITE)

    lon_b, lat_b = panel_label_corner_lon_lat(FOX_REGION)
    fig.text(x=lon_b, y=lat_b, text="B", font=LABEL_PANEL, justify="LB")
    fig.text(
        x=0.5 * (FOX_REGION[0] + FOX_REGION[1]),
        y=FOX_REGION[2] + 0.013 * (FOX_REGION[3] - FOX_REGION[2]),
        text="Basemap: Esri",
        font="16p,Helvetica,black",
        justify="CB",
        fill=BASEMAP_CREDIT_FILL,
        clearance=BASEMAP_CREDIT_CLEARANCE,
        pen=BASEMAP_CREDIT_BOX_PEN,
    )

fig.plot(
    region=[0, W, 0, H],
    projection=f"X{W}c/{H}c",
    x=[sx1, xg1, np.nan, sx2, xg2],
    y=[sy1, yg1, np.nan, sy2, yg2],
    pen="0.4p,black",
)

fig.savefig(OUT_PNG, dpi=300)

print(OUT_PNG)
